In [2]:
import torch
from torch.utils.data import DataLoader, random_split
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score
import os
import random

from data.preprocessing.pipeline import Pipeline
from data.datasets.MLP_Dataset import MultimodalDataset  
from models.MLP4 import MultimodalModel  
from models.MLP4 import FocalLoss
from utils.paths import WEIGHTS_DIR

from torch.utils.data import DataLoader, random_split
from torchvision import transforms

In [3]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

In [4]:
random_state = 42
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
epochs = 30
batch_size = 64
learning_rate = 1e-4
val_size = 0.2
patience = 10
checkpoint_path="best_model.pth"
save_dir_path = os.path.join(WEIGHTS_DIR, 'MLP2')

set_seed(random_state)

In [5]:
pipeline = Pipeline(feature_type="rdkit+dft", use_pca_rdkit=False, use_pca_dft=False, remove_zero_var=True)
train_df, test_df = pipeline.get_processed_data()

Extracting raw features:   0%|          | 0/2064 [00:00<?, ?it/s]

[18:49:16] DEPRECATION WARNING: please use MorganGenerator
[18:49:16] DEPRECATION WARNING: please use MorganGenerator
[18:49:16] DEPRECATION WARNING: please use MorganGenerator
[18:49:16] DEPRECATION WARNING: please use MorganGenerator
[18:49:16] DEPRECATION WARNING: please use MorganGenerator
[18:49:16] DEPRECATION WARNING: please use MorganGenerator
[18:49:16] DEPRECATION WARNING: please use MorganGenerator
Extracting raw features:   0%|          | 7/2064 [00:00<00:30, 67.20it/s][18:49:16] DEPRECATION WARNING: please use MorganGenerator
[18:49:16] DEPRECATION WARNING: please use MorganGenerator
[18:49:16] DEPRECATION WARNING: please use MorganGenerator
[18:49:16] DEPRECATION WARNING: please use MorganGenerator
[18:49:16] DEPRECATION WARNING: please use MorganGenerator
[18:49:16] DEPRECATION WARNING: please use MorganGenerator
[18:49:16] DEPRECATION WARNING: please use MorganGenerator
[18:49:16] DEPRECATION WARNING: please use MorganGenerator
[18:49:16] DEPRECATION WARNING: please use

In [25]:
def train_image_model(  # Renamed to train_multimodal_model for clarity, but kept name as per your code; update calls accordingly if needed
    images, df, epochs=20, batch_size=64, lr=1e-3,  # Changed labels to df, removed labels arg since df has them
    val_size=0.2, patience=5, checkpoint_path="best_image_model.pth",
    save_dir_path=".", device='cpu'
):
    os.makedirs(save_dir_path, exist_ok=True)

    mean = [0.485, 0.456, 0.406]
    std = [0.229, 0.224, 0.225]

    train_transform = transforms.Compose([
        transforms.ToTensor(),  # HWC -> CHW, [0,1]
        transforms.Normalize(mean=mean, std=std)
    ])

    val_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std)
    ])

    full_dataset = MultimodalDataset(images, df, train=True, transform=None)  # Use new dataset with df
    n_total = len(full_dataset)
    n_val = int(n_total * val_size)
    n_train = n_total - n_val
    train_subset, val_subset = random_split(full_dataset, [n_train, n_val])

    for subset in [train_subset, val_subset]:
        subset.dataset.transform = train_transform if subset == train_subset else val_transform

    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)

    # Updated to MultimodalModel; compute tabular_dim from dataset
    tabular_dim = full_dataset.tabular.shape[1]
    model = MultimodalModel(num_classes=len(np.unique(full_dataset.labels)), tabular_dim=tabular_dim).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = FocalLoss(gamma=1.5)

    best_val_loss = float("inf")
    patience_counter = 0
    all_val_preds = []
    all_val_targets = []

    for epoch in range(epochs):
        model.train()
        train_losses = []

        for imgs, tabs, ys in train_loader:  # Unpack imgs, tabs, ys
            imgs = imgs.to(device)
            tabs = tabs.to(device)
            ys = ys.to(device)

            optimizer.zero_grad()
            logits = model(imgs, tabs)  # Pass both to model
            loss = criterion(logits, ys)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []
        all_preds = []
        all_targets = []

        with torch.no_grad():
            for imgs, tabs, ys in val_loader:
                imgs = imgs.to(device)
                tabs = tabs.to(device)
                ys = ys.to(device)

                logits = model(imgs, tabs)
                loss = criterion(logits, ys)
                val_losses.append(loss.item())

                preds = torch.argmax(logits, dim=1)
                all_preds.append(preds.cpu().numpy())
                all_targets.append(ys.cpu().numpy())

        train_loss = np.mean(train_losses)
        val_loss = np.mean(val_losses)
        print(f"Epoch {epoch+1}/{epochs} | train_loss={train_loss:.5f} | val_loss={val_loss:.5f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), os.path.join(save_dir_path, checkpoint_path))
            print(f"  → New best model saved! val_loss={best_val_loss:.5f}")
        else:
            patience_counter += 1
            print(f"  (patience {patience_counter}/{patience})")
            if patience_counter >= patience:
                print("\nEarly stopping triggered!")
                break

        all_val_preds = np.concatenate(all_preds)
        all_val_targets = np.concatenate(all_targets)

    macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
    print("\nFinal macro F1 on validation:", macro_f1)

    return model, macro_f1, best_val_loss

In [11]:
import os
import numpy as np
import torch
from torch.utils.data import DataLoader, WeightedRandomSampler, random_split
from torchvision import transforms
from sklearn.metrics import f1_score, classification_report
import torch.nn.functional as F

def train_multimodal_model(
    images,
    df,
    epochs=20,
    batch_size=64,
    lr_head=1e-3,
    lr_backbone=1e-5,
    freeze_epochs=5,
    val_size=0.2,
    patience=5,
    checkpoint_path="best_model.pth",
    save_dir_path=".",
    device="cpu"
):
    os.makedirs(save_dir_path, exist_ok=True)

    # ============================
    #  Data augmentation
    # ============================
    mean = [0.485, 0.456, 0.406]
    std = [0.229, 0.224, 0.225]

    train_transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.1, hue=0.05),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])

    val_transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])

    # ============================
    # Dataset / Split
    # ============================
    full_dataset = MultimodalDataset(images, df, train=True, transform=None)
    n_total = len(full_dataset)
    n_val = int(n_total * val_size)
    n_train = n_total - n_val
    train_subset, val_subset = random_split(full_dataset, [n_train, n_val])

    # assign transforms for subsets
    train_subset.dataset.transform = train_transform
    val_subset.dataset.transform = val_transform

    # ============================
    # WeightedRandomSampler (compute per-train-index weights)
    # ============================
    if hasattr(full_dataset, "labels"):
        all_labels = np.array(full_dataset.labels)
    else:
        all_labels = np.array(df["Label"].values)

    # Проверяем, что метки целочисленные
    all_labels = all_labels.astype(int)
    
    classes, counts = np.unique(all_labels, return_counts=True)
    num_classes = len(classes)
    
    print(f"Class distribution: {dict(zip(classes, counts))}")
    
    # Более безопасное вычисление весов
    total_samples = len(all_labels)
    class_weights = total_samples / (num_classes * counts)
    class_weight_map = {c: w for c, w in zip(classes, class_weights)}
    
    print(f"Class weights: {class_weight_map}")

    # build sample weights for all indices, then pick train indices
    sample_weights_all = np.array([class_weight_map[int(l)] for l in all_labels])
    train_indices = train_subset.indices
    sample_weights_train = sample_weights_all[train_indices]

    sampler = WeightedRandomSampler(
        weights=sample_weights_train,
        num_samples=len(sample_weights_train),
        replacement=True
    )

    train_loader = DataLoader(train_subset, batch_size=batch_size, sampler=sampler, num_workers=4, pin_memory=True)
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

    # ============================
    # Model
    # ============================
    tabular_dim = full_dataset.tabular.shape[1]

    model = MultimodalModel(
        tabular_dim=tabular_dim,
        num_classes=num_classes,
        freeze_backbone=True
    ).to(device)

    # optimizer: initially don't include backbone params (frozen)
    optimizer = torch.optim.Adam([
        {"params": model.image_branch.fc.parameters(), "lr": lr_head},
        {"params": model.tabular_branch.parameters(), "lr": lr_head},
        {"params": model.classifier.parameters(), "lr": lr_head},
    ], weight_decay=1e-4)

    # Более мягкий Focal Loss
    weight_tensor = torch.tensor([class_weight_map.get(c, 1.0) for c in range(num_classes)], dtype=torch.float32).to(device)
    criterion = FocalLoss(gamma=1.0, weight=weight_tensor)  # Уменьшили gamma

    best_val_f1 = 0.0  # Теперь отслеживаем F1 вместо loss
    patience_counter = 0

    print("Training started...")

    for epoch in range(epochs):

        # ===== UNFREEZE BACKBONE =====
        if epoch == freeze_epochs:
            print(f"\n>> Unfreezing ResNet18 backbone at epoch {epoch}!")
            model.unfreeze_backbone()
            optimizer.add_param_group({
                "params": model.image_branch.features.parameters(),
                "lr": lr_backbone
            })

        # ===== TRAIN =====
        model.train()
        train_losses = []
        all_train_preds = []
        all_train_targets = []

        for imgs, tabs, ys in train_loader:
            imgs, tabs, ys = imgs.to(device), tabs.to(device), ys.to(device)

            optimizer.zero_grad()
            logits = model(imgs, tabs)
            loss = criterion(logits, ys)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())
            
            # Собираем предсказания для метрик
            preds = torch.argmax(logits, dim=1)
            all_train_preds.extend(preds.cpu().numpy())
            all_train_targets.extend(ys.cpu().numpy())

        # ===== VALIDATION =====
        model.eval()
        val_losses = []
        all_val_preds = []
        all_val_targets = []

        with torch.no_grad():
            for imgs, tabs, ys in val_loader:
                imgs, tabs, ys = imgs.to(device), tabs.to(device), ys.to(device)
                logits = model(imgs, tabs)
                loss = criterion(logits, ys)
                val_losses.append(loss.item())

                preds = torch.argmax(logits, dim=1)
                all_val_preds.extend(preds.cpu().numpy())
                all_val_targets.extend(ys.cpu().numpy())

        # Вычисляем метрики
        train_loss = np.mean(train_losses)
        val_loss = np.mean(val_losses)
        
        train_f1 = f1_score(all_train_targets, all_train_preds, average='macro', zero_division=0)
        val_f1 = f1_score(all_val_targets, all_val_preds, average='macro', zero_division=0)
        
        train_acc = np.mean(np.array(all_train_preds) == np.array(all_train_targets))
        val_acc = np.mean(np.array(all_val_preds) == np.array(all_val_targets))

        print(f"Epoch {epoch+1}/{epochs} | "
              f"train_loss={train_loss:.5f} | val_loss={val_loss:.5f} | "
              f"train_f1={train_f1:.4f} | val_f1={val_f1:.4f} | "
              f"train_acc={train_acc:.4f} | val_acc={val_acc:.4f}")

        # Сохраняем модель по лучшему F1-score
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            patience_counter = 0
            torch.save(model.state_dict(), os.path.join(save_dir_path, checkpoint_path))
            print(f"  → New best model saved (val_f1={val_f1:.5f})")
        else:
            patience_counter += 1
            print(f"  (patience {patience_counter}/{patience})")
            if patience_counter >= patience:
                print("\nEarly stopping triggered!")
                break

    # Финальная оценка
    print("\n" + "="*50)
    print("FINAL RESULTS:")
    print(f"Best validation F1-macro: {best_val_f1:.5f}")
    
    # Детальный отчет по классам
    if len(all_val_preds) > 0:
        print("\nClassification Report:")
        print(classification_report(all_val_targets, all_val_preds, zero_division=0))
    
    return model, best_val_f1, val_loss

In [26]:
def predict(model, images, df, batch_size=64, device='cpu'):  # Added df for tabular
    model.eval()
    dataset = MultimodalDataset(images, df, train=False)  # train=False for inference
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    all_preds = []
    all_ids = []  # To collect ids if needed
    with torch.no_grad():
        for imgs, tabs, ids in loader:  # Unpack imgs, tabs, ids
            imgs = imgs.to(device)
            tabs = tabs.to(device)
            logits = model(imgs, tabs)
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            all_preds.append(preds)
            all_ids.append(ids.numpy())  # Assuming ids are tensors; adjust if needed

    return np.concatenate(all_preds)  # Return preds; use all_ids if needed for df

In [12]:
train_images, test_images = pipeline.get_images()  # train_images: (2064, 244, 244, 3)

model, macro_f1, best_val_loss = train_multimodal_model(
    images=train_images,
    df=train_df,                 # содержит SMILES, tabular, label
    epochs=30,
    batch_size=64,

    # learning rates:
    lr_head=1e-3,               # для MLP + FC ResNet
    lr_backbone=1e-5,           # для разморозки backbone после freeze_epochs

    freeze_epochs=6,            # 5–8 эпох — из рекомендаций
    val_size=0.2,
    patience=10,

    checkpoint_path="best_multimodal_model.pth",
    save_dir_path=save_dir_path,
    device=device
)

print(f"Training finished. Best validation loss: {best_val_loss:.5f}, macro F1: {macro_f1:.5f}")

Class distribution: {np.int64(0): np.int64(530), np.int64(1): np.int64(296), np.int64(2): np.int64(220), np.int64(3): np.int64(402), np.int64(4): np.int64(286), np.int64(5): np.int64(43), np.int64(6): np.int64(85), np.int64(7): np.int64(202)}
Class weights: {np.int64(0): np.float64(0.4867924528301887), np.int64(1): np.float64(0.8716216216216216), np.int64(2): np.float64(1.1727272727272726), np.int64(3): np.float64(0.6417910447761194), np.int64(4): np.float64(0.9020979020979021), np.int64(5): np.float64(6.0), np.int64(6): np.float64(3.0352941176470587), np.int64(7): np.float64(1.2772277227722773)}
Training started...
Epoch 1/30 | train_loss=1.94373 | val_loss=1.28643 | train_f1=0.2671 | val_f1=0.2580 | train_acc=0.3354 | val_acc=0.2524
  → New best model saved (val_f1=0.25796)
Epoch 2/30 | train_loss=1.15788 | val_loss=1.23228 | train_f1=0.4218 | val_f1=0.2750 | train_acc=0.4746 | val_acc=0.2694
  → New best model saved (val_f1=0.27496)
Epoch 3/30 | train_loss=1.08000 | val_loss=1.26528

In [27]:
train_images, test_images = pipeline.get_images()  # train_images: (2064, 244, 244, 3)
# Removed train_labels extraction; now using train_df directly

model, macro_f1, best_val_loss = train_image_model(
    images=train_images,
    df=train_df,  # Changed from labels to df
    epochs=epochs,
    batch_size=batch_size,
    lr=learning_rate,
    val_size=val_size,
    patience=patience,
    checkpoint_path=checkpoint_path,
    save_dir_path=save_dir_path,
    device=device
)

print(f"Training finished. Best validation loss: {best_val_loss:.5f}, macro F1: {macro_f1:.5f}")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\edtop/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:05<00:00, 8.95MB/s]


Epoch 1/30 | train_loss=1.35091 | val_loss=1.24849
  → New best model saved! val_loss=1.24849
Epoch 2/30 | train_loss=0.95888 | val_loss=1.15791
  → New best model saved! val_loss=1.15791
Epoch 3/30 | train_loss=0.66630 | val_loss=1.18479
  (patience 1/10)
Epoch 4/30 | train_loss=0.40847 | val_loss=1.33223
  (patience 2/10)
Epoch 5/30 | train_loss=0.23482 | val_loss=1.41503
  (patience 3/10)
Epoch 6/30 | train_loss=0.15399 | val_loss=1.47721
  (patience 4/10)
Epoch 7/30 | train_loss=0.11208 | val_loss=1.67699
  (patience 5/10)
Epoch 8/30 | train_loss=0.08095 | val_loss=1.53271
  (patience 6/10)
Epoch 9/30 | train_loss=0.05981 | val_loss=1.71071
  (patience 7/10)
Epoch 10/30 | train_loss=0.03783 | val_loss=1.78933
  (patience 8/10)
Epoch 11/30 | train_loss=0.02942 | val_loss=1.71356
  (patience 9/10)
Epoch 12/30 | train_loss=0.02547 | val_loss=1.86464
  (patience 10/10)

Early stopping triggered!

Final macro F1 on validation: 0.366737532906932
Training finished. Best validation loss: 1

In [ ]:
model.load_state_dict(torch.load(os.path.join(save_dir_path, checkpoint_path)))
predictions = predict(model, test_images, test_df, batch_size=batch_size, device=device)  # Added test_df